[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S06/S06_02_limpieza.ipynb)

# S06 · 02 — Limpieza de datos: encontrar defectos y sus causas

**Tiempo estimado:** 70 min · **Sprint 6** · Módulo M2 (UD 2.4)

**Contexto TurisData.** El cliente nos envía un segundo fichero, `reservas_sucio.csv`,
exportado "a mano" desde varios sistemas. Antes de usarlo hay que **localizar los defectos,
entender de dónde vienen y corregirlos con reglas claras**. Limpiar no es "borrar lo feo":
cada decisión (eliminar, corregir, dejar a nulo) debe estar justificada.

**Al terminar sabrás:**
1. Diagnosticar nulos, duplicados, tipos incorrectos, categorías inconsistentes y fechas mezcladas.
2. Distinguir un valor **imposible** (error) de un valor **raro** (posible outlier real).
3. Escribir una función de limpieza reproducible.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEMILLA = 42
sns.set_theme(style="whitegrid")

## 1. Diagnóstico inicial

**Idea.** Nunca limpies a ciegas: primero mide. Guardamos siempre el original (`crudo`)
y trabajamos sobre una copia, para poder comparar antes/después.

In [ ]:
crudo = pd.read_csv(dato("reservas_sucio.csv"))
print(crudo.shape)
crudo.info()

In [ ]:
def resumen_calidad(tabla):
    return pd.DataFrame({
        "tipo": tabla.dtypes.astype(str),
        "nulos": tabla.isna().sum(),
        "pct_nulos": (tabla.isna().mean() * 100).round(2),
        "distintos": tabla.nunique(),
    })

resumen_calidad(crudo)

Pistas que ya se ven: `precio_noche` es de tipo `object` (¡debería ser numérico!),
`adultos` es decimal (por los nulos), las fechas son texto y hay columnas con nulos.
Vamos a ir defecto a defecto.

## 2. Duplicados

**Idea.** Un duplicado exacto suele venir de una exportación repetida o de un doble envío.
Si no se quita, esa reserva "pesa" dos veces y, peor, puede acabar en train **y** en test.

In [ ]:
print("Filas duplicadas exactas:", crudo.duplicated().sum())
crudo[crudo.duplicated(keep=False)].sort_values("id_reserva").head(6)

**Ejercicio 1.** Guarda en `n_duplicados` cuántas filas están repetidas y crea `d1`,
una copia de `crudo` **sin duplicados** y con el índice reiniciado.

In [ ]:
# TODO: duplicated().sum() y drop_duplicates().reset_index(drop=True)
...
assert n_duplicados == 45, "Cuenta las filas duplicadas con crudo.duplicated().sum()"
assert len(d1) == 1500, "Tras quitar duplicados deben quedar 1500 reservas"
assert d1["id_reserva"].is_unique, "id_reserva debería ser único tras deduplicar"

## 3. Tipos incorrectos: números escritos como texto

**Idea.** Algunos precios llevan **coma decimal** ("84,36"), típica de una exportación de
Excel en español. Por eso toda la columna es texto. Se corrige reemplazando la coma y
convirtiendo con `pd.to_numeric`. Con `errors="coerce"` lo que no se pueda convertir pasa
a nulo (mejor que romper el proceso, pero **hay que contarlo**).

In [ ]:
con_coma = d1["precio_noche"].astype(str).str.contains(",").sum()
print("Precios con coma decimal:", con_coma)

**Ejercicio 2.** Crea `d2` (copia de `d1`) donde `precio_noche` sea numérico (float),
cambiando la coma por punto. Guarda en `nulos_precio` el número de nulos resultantes.

In [ ]:
# TODO: .astype(str).str.replace(",", ".") y pd.to_numeric(..., errors="coerce")
...
assert d2["precio_noche"].dtype == float, "precio_noche debe ser float"
assert 60 <= nulos_precio <= 90, "Los nulos de precio deberían ser los ya existentes (unos 5 %); si salen muchos más, revisa la conversión"

## 4. Fechas en formatos mezclados

**Idea.** `fecha_reserva` mezcla `2024-11-19` (ISO) con `19/11/2024` (día/mes/año).
Ojo con la trampa: `05/11/2024` podría leerse como 5 de noviembre o 11 de mayo.
**No adivines**: como sabemos que el formato "con barras" es día/mes/año, se lee cada
grupo con su formato y se combinan.

In [ ]:
iso = pd.to_datetime(d2["fecha_reserva"], format="%Y-%m-%d", errors="coerce")
espanol = pd.to_datetime(d2["fecha_reserva"], format="%d/%m/%Y", errors="coerce")
print("ISO:", iso.notna().sum(), "· dd/mm/aaaa:", espanol.notna().sum(), "· sin leer:", (iso.isna() & espanol.isna()).sum())

**Ejercicio 3.** Crea `d3` (copia de `d2`) con `fecha_reserva` y `fecha_llegada` como fechas
(`datetime64`). Para `fecha_reserva` combina los dos formatos (pista: `iso.fillna(espanol)`).

In [ ]:
# TODO: iso.fillna(espanol) para fecha_reserva; pd.to_datetime para fecha_llegada
...
assert str(d3["fecha_reserva"].dtype).startswith("datetime"), "fecha_reserva debe ser datetime"
assert d3["fecha_reserva"].isna().sum() == 0, "No debe quedar ninguna fecha sin leer"
assert d3["fecha_reserva"].min().year == 2024, "Comprueba que no has intercambiado día y mes"

## 5. Categorías inconsistentes

**Idea.** "Directo", "directo " y "DIRECTO" son la misma cosa para una persona pero
tres categorías distintas para el ordenador. Se normaliza: quitar espacios, pasar a
minúsculas, unificar separadores y, si hace falta, un diccionario de equivalencias.

In [ ]:
print(d3["canal"].value_counts().to_string())
print()
print(d3["pais_origen"].value_counts().to_string())

**Ejercicio 4.** Normaliza `canal`: sin espacios sobrantes, en minúsculas y con `_` en lugar
de espacios. Después normaliza `pais_origen` con el diccionario `equivalencias` para que
queden solo los 8 valores válidos (`ES, UK, DE, NL, IT, FR, Nordicos, otros`).
Guarda el resultado en `d4`.

In [ ]:
equivalencias = {"es": "ES", "españa": "ES", "uk": "UK", "reino unido": "UK", "de": "DE",
                 "nl": "NL", "it": "IT", "fr": "FR", "nordicos": "Nordicos", "otros": "otros"}
# TODO: .str.strip().str.lower().str.replace(" ", "_") y .str.strip().str.lower().map(equivalencias)
...
assert sorted(d4["canal"].unique()) == ["agencia", "directo", "empresa", "web_intermediaria"], "Canal: deben quedar 4 valores"
assert d4["pais_origen"].nunique() == 8, "País: deben quedar 8 valores (los nulos no cuentan)"
assert d4["pais_origen"].isna().sum() == d3["pais_origen"].isna().sum(), "El mapeo no debe crear nulos nuevos: revisa el diccionario"

## 6. Valores imposibles y *outliers*

**Idea.** Hay dos clases de valores extraños:
- **Imposibles** (regla de negocio rota): precio ≤ 0, noches ≤ 0, antelación negativa. Son
  **errores**: se convierten en nulo (y se imputarán después) o se corrigen si se puede.
- **Raros pero posibles** (una suite de 600 €): son *outliers* y **no se borran sin motivo**.

Una herramienta rápida para detectarlos es la regla del **rango intercuartílico (IQR)**:
sospechoso lo que queda fuera de `[Q1 − 1,5·IQR, Q3 + 1,5·IQR]`.

In [ ]:
print(d4[["precio_noche", "noches", "antelacion_dias"]].describe().round(1))
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.boxplot(x=d4["precio_noche"], ax=ejes[0], color="#8ecae6")
ejes[0].set(title="Precio con outliers", xlabel="Precio por noche (€)")
sns.boxplot(x=d4.loc[d4["precio_noche"].between(1, 1000), "precio_noche"], ax=ejes[1], color="#8ecae6")
ejes[1].set(title="Precio sin los valores imposibles", xlabel="Precio por noche (€)")
plt.tight_layout()
plt.show()

**Ejercicio 5.** Crea `d5` (copia de `d4`) aplicando estas reglas de negocio, convirtiendo
en `np.nan` (no borrando filas) los valores que las incumplan:
- `precio_noche` debe estar entre 20 y 1000 €,
- `noches` debe estar entre 1 y 60.

(La antelación negativa la trataremos en la sección 7, cruzándola con las fechas.)

Guarda en `n_imposibles` cuántos valores has convertido en nulo en total.

In [ ]:
# TODO: usa .where(condición) o .mask(~condición); cuenta los nulos nuevos con isna().sum()
...
assert d5["precio_noche"].dropna().between(20, 1000).all(), "Quedan precios fuera de rango"
assert (d5["noches"].dropna() >= 1).all(), "Quedan noches <= 0"
assert 8 <= n_imposibles <= 14, "Deberían ser unos 10-14 valores imposibles (precios y noches). Revisa las reglas"

## 7. Consistencia entre columnas

**Idea.** Además de mirar cada columna, hay que cruzarlas. Aquí
`antelacion_dias` debe ser igual a `fecha_llegada − fecha_reserva` en días.
Si no cuadra, la fuente de verdad son **las fechas** (más difíciles de falsear por error).

In [ ]:
antelacion_calculada = (d5["fecha_llegada"] - d5["fecha_reserva"]).dt.days
incoherentes = antelacion_calculada != d5["antelacion_dias"]
print("Antelaciones negativas:", (d5["antelacion_dias"] < 0).sum())
print("Antelaciones que no cuadran con las fechas:", incoherentes.sum())
# Las fechas mandan: recalculamos toda la columna desde ellas
d5["antelacion_dias"] = antelacion_calculada
print("Mínimo tras recalcular:", d5["antelacion_dias"].min())

## 8. Nulos: ¿qué hacemos con ellos?

**Idea.** Antes de imputar, pregunta **por qué falta** el dato:
- Si falta al azar y son pocos → imputar (mediana en numéricas, moda o "desconocido" en categóricas).
- Si la ausencia tiene significado (p. ej., país no informado en reservas por teléfono) → dejar una categoría `"desconocido"` o una columna indicadora.
- Si falta casi todo en una columna → valorar quitar la columna.

**Regla de oro:** los métodos que *aprenden* de los datos (mediana, moda, escalado) se
ajustan **solo con el conjunto de entrenamiento**, dentro de un *pipeline*. Lo veremos en
la práctica 3. Aquí solo contamos y documentamos.

In [ ]:
resumen_calidad(d5).query("nulos > 0")

## 9. Mini-reto: tu función `limpiar_reservas`

Junta todo lo anterior en una función que reciba el DataFrame crudo y devuelva el limpio.
Debe ser **reproducible** (misma entrada → misma salida) y **no modificar** el original.
Pasos: duplicados → precio a número → fechas → categorías → imposibles a nulo →
antelación recalculada. Devuelve el resultado en `limpio` llamando a tu función.

In [ ]:
# TODO: encadena los pasos de los ejercicios 1-5 y la sección 7 dentro de la función
...
assert len(crudo) == 1545, "La función no debe modificar el DataFrame original"
assert limpio.shape[0] == 1500, "Deben quedar 1500 filas"
assert limpio["id_reserva"].is_unique
assert limpio["canal"].nunique() == 4 and limpio["pais_origen"].nunique() == 8
assert limpio["precio_noche"].dtype == float
assert (limpio["antelacion_dias"] >= 0).all() and (limpio["noches"].dropna() > 0).all()
assert str(limpio["fecha_reserva"].dtype).startswith("datetime")

In [ ]:
# Comparación antes/después
comparacion = pd.DataFrame({
    "antes": [len(crudo), crudo.duplicated().sum(), crudo["canal"].nunique(), crudo["pais_origen"].nunique()],
    "después": [len(limpio), limpio.duplicated().sum(), limpio["canal"].nunique(), limpio["pais_origen"].nunique()],
}, index=["filas", "duplicados", "valores distintos de canal", "valores distintos de país"])
comparacion

## 10. Para reflexionar

1. ¿Por qué convertimos los imposibles en nulo en lugar de borrar la fila entera?
2. Un precio de 600 € en una suite en agosto, ¿es un error? ¿Qué harías con él?
3. Si el 40 % de una columna estuviera vacío, ¿qué opciones hay y qué riesgos tiene cada una?
4. ¿Qué pregunta harías **al cliente** para saber por qué hay precios con coma y fechas mezcladas?

## 11. Resumen: qué has aprendido
- Diagnostica antes de limpiar y conserva siempre el original.
- Defectos típicos: duplicados, tipos, formatos de fecha, categorías, imposibles, nulos e incoherencias entre columnas.
- Imposible ≠ raro: los errores se corrigen; los raros se investigan.
- La imputación y el escalado se harán **dentro del pipeline** para evitar fugas.